Импортируем библиотеки

In [18]:
import nltk
import random
import numpy as np
import pandas as pd
import pprint, time
from sklearn.model_selection import train_test_split

Скачиваем данные из Github

In [19]:
!wget https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
!wget https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt

--2024-11-07 21:28:34--  https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7626752 (7.3M) [text/plain]
Saving to: ‘GSD_train.txt.1’

GSD_train.txt.1     100%[===================>]   7.27M   623KB/s    in 13s     

2024-11-07 21:28:47 (584 KB/s) - ‘GSD_train.txt.1’ saved [7626752/7626752]

--2024-11-07 21:28:48--  https://raw.githubusercontent.com/appling2024/data/refs/heads/main/GSD_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 81386 (79K) [t

In [20]:
with open("GSD_train.txt", encoding='utf-8') as f:
  data = f.read()

In [21]:
sent = data.split('\n\n')

Представляем данные в виде списка, который содержит другие списки

In [22]:
tokens = []
for sentence in sent:
    s = []
    for word in sentence.split('\n'):
        if word:
            token = (word.split()[1], word.split()[3])
        s.append(token)
    tokens.append(s)

Подготовка к обучению HMM

In [23]:
train_set,test_set =train_test_split(tokens, train_size=0.80, test_size=0.20, random_state = 101)

Создаем список размеченных обучающих и тестовых данных

In [24]:
train_tagged_words = [ tup for sent in train_set for tup in sent ]
test_tagged_words = [ tup for sent in test_set for tup in sent ]

Рассчитываем вероятности эмиссии
они определяют вероятность увидеть определенную наблюдаемую переменную при заданном значении для скрытых переменных,
т.е. эта функция возвращает то, сколько всего раз тег встречался в выборке и сколько раз определенное слово встречалось с этим тегом

In [25]:
def word_given_tag(word, tag, train_bag=train_tagged_words):
    tag_list = [pair for pair in train_bag if pair[1]==tag] # список всех слов определенного тега
    count_tag = len(tag_list)# общее число появления тегов в обучающей выборке
    w_given_tag_list = [pair[0] for pair in tag_list if pair[0]==word] # cписок, который состоит из всех вхождений данного слова, которые были помечены определенным тегом
    count_w_given_tag = len(w_given_tag_list) # подсчитываем общее количество раз, когда слово встречалось с тегом

    return (count_w_given_tag, count_tag)

Рассчитываем вероятность переходов

In [26]:
def t2_given_t1(t2, t1, train_bag=train_tagged_words): # t1 = t2, это пронумерованный список тегов
    tags = [pair[1] for pair in train_bag] # список всех тегов по порядку (просто из общего спсика размеченных слов мы убираем слова, получаем последовательность тегов)
    count_t1 = len([t for t in tags if t==t1])
    count_t2_t1 = 0
    for index in range(len(tags)-1):
        if tags[index]==t1 and tags[index+1] == t2: # если при теге t1 следующий тег = t2, то осуществляется переход
            count_t2_t1 += 1
    return (count_t2_t1, count_t1) # сколько всего раз встречается данный (второе значение), сколько раз данный тег переходит в другой тег (первое значение)

создаем матрицу тегов t x t, где t - номер тега
матрица(i, j) представляет вероятность перехода P(i-й тег переходит в j-й тег)

In [28]:
tags = list(set([pair[1] for pair in train_tagged_words]))
tags_matrix = np.zeros((len(tags), len(tags)), dtype='float32')
for i, t1 in enumerate(list(tags)):
    for j, t2 in enumerate(list(tags)):
        tags_matrix[i, j] = t2_given_t1(t2, t1)[0]/t2_given_t1(t2, t1)[1]

print(tags_matrix)

[[8.16326495e-03 1.26530617e-01 0.00000000e+00 1.63265299e-02
  1.95918363e-02 2.44897953e-03 3.26530612e-03 2.44897953e-03
  7.14285731e-01 1.06122447e-02 3.26530612e-03 2.69387756e-02
  1.63265306e-03 9.79591813e-03 5.71428565e-03 4.89795916e-02]
 [2.16517178e-03 8.56789351e-02 2.06206823e-04 8.56789351e-02
  1.26817198e-02 2.19610278e-02 2.68068863e-03 9.27930698e-04
  7.36364603e-01 1.95896486e-03 1.03103412e-04 2.34044753e-02
  2.98999902e-03 1.93834417e-02 1.54655124e-03 2.26827501e-03]
 [6.94444450e-03 7.63888881e-02 4.16666679e-02 1.80555552e-01
  9.02777761e-02 6.94444450e-03 0.00000000e+00 9.02777761e-02
  2.63888896e-01 1.31944448e-01 0.00000000e+00 8.33333358e-02
  0.00000000e+00 2.08333340e-02 6.94444450e-03 0.00000000e+00]
 [1.34491455e-02 1.24264501e-01 1.12076208e-03 1.18310452e-01
  9.91874486e-02 5.16251065e-02 4.73521985e-02 2.71784812e-02
  1.65802747e-01 2.93499585e-02 2.71084346e-02 1.51513025e-01
  5.18352492e-03 9.44242105e-02 3.46035287e-02 9.45643056e-03]
 [2.

In [29]:
tags_df = pd.DataFrame(tags_matrix, columns = list(tags), index=list(tags))
display(tags_df)

,DET,ADJ,SYM,PUNCT,VERB,CCONJ,ADV,X,NOUN,NUM,SCONJ,ADP,AUX,PROPN,PRON,PART
DET,0.008163,0.126531,0.000000,0.016327,0.019592,0.002449,0.003265,0.002449,0.714286,0.010612,0.003265,0.026939,0.001633,0.009796,0.005714,0.048980
ADJ,0.002165,0.085679,0.000206,0.085679,0.012682,0.021961,0.002681,0.000928,0.736365,0.001959,0.000103,0.023404,0.002990,0.019383,0.001547,0.002268
SYM,0.006944,0.076389,0.041667,0.180556,0.090278,0.006944,0.000000,0.090278,0.263889,0.131944,0.000000,0.083333,0.000000,0.020833,0.006944,0.000000
PUNCT,0.013449,0.124265,0.001121,0.118310,0.099187,0.051625,0.047352,0.027178,0.165803,0.029350,0.027108,0.151513,0.005184,0.094424,0.034604,0.009456
VERB,0.028060,0.149552,0.000448,0.079254,0.049403,0.010000,0.035522,0.002687,0.234179,0.038358,0.000597,0.300149,0.003731,0.024776,0.026418,0.016866
CCONJ,0.024553,0.177695,0.001248,0.033708,0.131086,0.027882,0.061590,0.011236,0.257179,0.026217,0.005826,0.101956,0.007074,0.085726,0.019975,0.027050
ADV,0.013498,0.128234,0.000562,0.116985,0.288526,0.021372,0.048931,0.003375,0.061867,0.032621,0.006187,0.165354,0.013498,0.013498,0.036558,0.048931
X,0.000805,0.008052,0.006441,0.432367,0.037037,0.028180,0.003221,0.406602,0.006441,0.030596,0.000000,0.020934,0.007246,0.005636,0.000000,0.006441
NOUN,0.010172,0.111331,0.001319,0.334040,0.083969,0.042196,0.012386,0.009325,0.151549,0.012904,0.000188,0.128662,0.011962,0.075681,0.007676,0.006640
NUM,0.004207,0.093750,0.042067,0.179087,0.015024,0.016827,0.001202,0.006010,0.549880,0.013822,0.000000,0.067308,0.003005,0.004207,0.000601,0.003005


Алгоритм Витерби

In [30]:
def Viterbi(words, train_bag=train_tagged_words): # первый аргумент - список слов, которым нам нужно присвоить тег, второй - размеченная обучающая выборка
    state = []
    T = list(set([pair[1] for pair in train_bag])) # список всех возможных тегов

    for key, word in enumerate(words): # пронумеровываем список слов, чтобы можно было смотреть на предыдущее
        p = [] # инициализируем список столбцов вероятности для каждого наблюдения
        for tag in T: # считаем вероятность перехода
            if key == 0: # если предыдущего тега нет, то рассматриваем вероятность появления слова после знака препинания, в начале предложения
                transition_p = tags_df.loc['PUNCT', tag]
            else: # если предыдущий тег есть, то рассматриваем его
                transition_p = tags_df.loc[state[-1], tag]

            # вычисляем вероятности выбросов и состояний
            emission_p = word_given_tag(words[key], tag)[0]/word_given_tag(words[key], tag)[1] # вероятность выбросов ( насколько вероятно, что конкретное слово будет иметь конкретный тег)
            state_probability = emission_p * transition_p  # вероятность состояний (умножаем веротность того, что слово в принципе встречалось с этим тегом на вероятность перехода)
            p.append(state_probability)


        pmax = max(p) # выбираем максимальное значение из полученного списка
        # получаем наиболее вероятную последовательность скрытых состояний
        state_max = T[p.index(pmax)] # записываем его индекс и находим его соответствие в списке тегов
        state.append(state_max)
    return list(zip(words, state))

In [31]:
# протестируем алгоритм Витерби на нескольких примерах предложений тестового набора данных
random.seed(1234)      #определяем случайное значение

# выбираем случайные 10 значений
rndom = [random.randint(1,len(test_set)) for x in range(10)]

# список из 10 предложений, на которых мы тестируем модель
test_run = [test_set[i] for i in rndom]

# список размеченных слов
test_run_base = [tup for sent in test_run for tup in sent]

# список неразмеченных слов
test_tagged_words = [tup[0] for sent in test_run for tup in sent]

Проверяем скорость и точность алгоритма

In [33]:
start = time.time()
tagged_seq = Viterbi(test_tagged_words)
end = time.time() # считаем сколько времени это заняло
difference = end-start

print("Время в секундах: ", difference)

# проходим по парам (разметка_HMM, разметка_эталон) и если совпали, сохраняем в список
# по существу, нам просто нужно получить количество совпадений
check = [i for i, j in zip(tagged_seq, test_run_base) if i == j]

accuracy = len(check)/len(tagged_seq) # считаем точность
print('Точность алгоритма Витерби, %: ',accuracy*100)

Время в секундах:  18.95428204536438
Точность алгоритма Витерби, %:  61.904761904761905


Ручная проверка 

In [34]:
for our, reference in list(zip(tagged_seq, test_run_base))[:12]:
  print('Наша: {0:20}\tЭталон: {1:20}'.format(str(our), str(reference)))

Наша: ('Двери', 'DET')    	Эталон: ('Двери', 'NOUN')   
Наша: ('из', 'ADP')       	Эталон: ('из', 'ADP')       
Наша: ('вагонов', 'NOUN') 	Эталон: ('вагонов', 'NOUN') 
Наша: ('поезда', 'NOUN')  	Эталон: ('поезда', 'NOUN')  
Наша: ('не', 'PART')      	Эталон: ('не', 'PART')      
Наша: ('открываются', 'DET')	Эталон: ('открываются', 'VERB')
Наша: ('.', 'PUNCT')      	Эталон: ('.', 'PUNCT')      
Наша: ('Назвав', 'DET')   	Эталон: ('Назвав', 'VERB')  
Наша: ('призванную', 'DET')	Эталон: ('призванную', 'VERB')
Наша: ('девушку', 'NOUN') 	Эталон: ('девушку', 'NOUN') 
Наша: ('в', 'ADP')        	Эталон: ('в', 'ADP')        
Наша: ('честь', 'NOUN')   	Эталон: ('честь', 'NOUN')   


Автоматизированная проверка 

In [50]:
mismatches = []
for our, reference in zip (tagged_seq, test_run_base):
    if our != reference:  
        mismatches.append((our[0], our[1], reference[1]))  
mismatches_df = pd.DataFrame(mismatches, columns=['Слово', 'Наша разметка', 'Эталонная разметка'])
mismatches_df.index = range(1, len(mismatches_df) + 1)
mismatches_df.to_excel('/Users/juliak/PycharmProjects/MSP/HMM/Mismatches.xlsx', index=False)
display (mismatches_df)

,Слово,Наша разметка,Эталонная разметка
1,Двери,DET,NOUN
2,открываются,DET,VERB
3,Назвав,DET,VERB
4,призванную,DET,VERB
5,кота,DET,NOUN
6,Танарот,DET,PROPN
7,хозяина,DET,NOUN
8,и,PART,CCONJ
9,слуги,DET,NOUN
10,Социал-демократической,DET,ADJ
